# models

> Data shapes for the transcription pipeline: run configuration + the run-manifest result containers.

The run manifest is the pipeline's durable output record: which sources were processed, how they were segmented, and where each segment's transcription landed (capability data DBs remain the authoritative text store; the manifest records the run's shape + provenance pointers). It is a deliberate proto-bundle — the CR-20 provenance-bundle infrastructure is expected to absorb/replace it.

In [ ]:
#| default_exp models

In [ ]:
#| export
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

In [ ]:
#| export
@dataclass
class PipelineConfig:
    """Configuration for one transcription pipeline run."""
    vad_plugin: str = "cjm-media-plugin-silero-vad"               # VAD capability instance id
    ffmpeg_plugin: str = "cjm-media-plugin-ffmpeg"                # Convert/segment capability instance id
    transcriber_plugin: str = "cjm-transcription-plugin-whisper"  # Transcription capability instance id
    max_segment_duration: float = 300.0  # Wall-clock cap per segment in seconds (pre-emptive cuts)
    sample_rate: int = 16000             # Model-input sample rate for the per-segment convert step
    channels: int = 1                    # Model-input channel count
    force: bool = False                  # Bypass capability-side caches (VAD + transcription)
    assume_yes: bool = False             # Auto-accept HITL seams (headless / corpus-generation mode)

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the run manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class SegmentRecord:
    """One transcribed segment of a source audio file."""
    index: int              # 0-based position within the source
    start: float            # Segment start in source-audio seconds
    end: float              # Segment end in source-audio seconds
    duration: float         # Wall-clock segment duration in seconds
    segment_path: str       # Cut audio file (source codec) from ffmpeg `segment_audio`
    model_input_path: str   # Model-ready WAV from the per-segment `convert` step
    job_id: str             # Provenance job id passed to the transcriber (keys its DB row)
    text: str               # Transcribed text
    metadata: Dict[str, Any] = field(default_factory=dict)  # Transcriber-reported metadata

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for the run manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class SourceResult:
    """Pipeline result for one source audio file."""
    source_path: str        # Original input audio path
    duration: float         # Source duration in seconds
    vad_chunk_count: int    # Number of speech chunks VAD detected
    batch_key: str          # ffmpeg `segment_audio` batch key linking the cut files
    segments: List[SegmentRecord] = field(default_factory=list)  # Ordered transcribed segments

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for the run manifest
        """Serialize to a plain dict with nested segments."""
        return {
            "source_path": self.source_path,
            "duration": self.duration,
            "vad_chunk_count": self.vad_chunk_count,
            "batch_key": self.batch_key,
            "segments": [s.to_dict() for s in self.segments],
        }

In [ ]:
#| export
@dataclass
class RunManifest:
    """Durable record of one pipeline run (proto-bundle; see CR-20)."""
    run_id: str                       # Unique run identifier
    created_at: float                 # Unix timestamp at run start
    config: Dict[str, Any]            # PipelineConfig snapshot
    plugins: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # instance_id -> {name, version, db_path}
    sources: List[SourceResult] = field(default_factory=list)         # Per-source results, input order

    FORMAT: str = field(default="cjm-transcription-core/run-manifest", repr=False)  # Manifest format tag
    VERSION: str = field(default="0.1.0", repr=False)                               # Manifest schema version

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
        """Serialize to a plain dict with nested sources."""
        return {
            "format": self.FORMAT,
            "version": self.VERSION,
            "run_id": self.run_id,
            "created_at": self.created_at,
            "config": self.config,
            "plugins": self.plugins,
            "sources": [s.to_dict() for s in self.sources],
        }

    def save(
        self,
        path: Union[str, Path],  # Destination JSON file (parent dirs created)
    ) -> Path:  # The written path
        """Write the manifest as pretty-printed JSON."""
        out = Path(path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(self.to_dict(), indent=2))
        return out

In [ ]:
#| export
def new_run_id() -> str:  # e.g. "run_20260607_153000_1a2b3c4d"
    """Generate a unique, sortable run id."""
    return f"run_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

In [ ]:
# Quick shape check (no plugins involved)
cfg = PipelineConfig()
m = RunManifest(run_id=new_run_id(), created_at=time.time(), config=cfg.to_dict())
assert m.to_dict()["format"] == "cjm-transcription-core/run-manifest"
assert m.to_dict()["config"]["max_segment_duration"] == 300.0
m.to_dict()["run_id"]

'run_20260607_001657_5dfd90e7'